# Model Cards con MLflow: Reutilizando Metadatos Registrados

Este notebook demuestra cómo crear **Model Cards** completas aprovechando toda la información que MLflow ha almacenado automáticamente durante el entrenamiento y registro del modelo. 

## ¿Qué son las Model Cards?

Las Model Cards son documentos que describen modelos de machine learning, proporcionando información crucial sobre:
- **Propósito del modelo** y casos de uso
- **Rendimiento** y métricas de evaluación  
- **Limitaciones** y sesgos conocidos
- **Consideraciones éticas** y de fairness
- **Datos de entrenamiento** y linaje
- **Información técnica** del modelo

## Ventajas de usar MLflow para Model Cards

✅ **Automatización**: MLflow registra automáticamente parámetros, métricas y artefactos  
✅ **Trazabilidad**: Conexión directa entre experimentos y documentación  
✅ **Consistencia**: Información siempre actualizada desde la fuente  
✅ **Gobernanza**: Mejor control de versiones y lineage de modelos

## 1. Configurar Conexión con MLflow

Primero configuramos la conexión con MLflow para acceder al registro de modelos donde está almacenado nuestro modelo entrenado.

In [ ]:
# Instalar dependencias necesarias
!pip install mlflow>=3.0.0 model-card-toolkit pandas matplotlib seaborn jinja2 --quiet

In [ ]:
import mlflow
from mlflow import MlflowClient
import pandas as pd
import json
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from jinja2 import Template

# Configurar el tracking server (usar el mismo ARN del notebook anterior)
tracking_server_arn = "your tracking server arn here"  # Reemplazar con tu ARN
mlflow.set_tracking_uri(tracking_server_arn)

# Crear cliente de MLflow
client = MlflowClient()

print("✅ Conexión establecida con MLflow")
print(f"📍 Tracking URI: {mlflow.get_tracking_uri()}")

# Listar modelos registrados disponibles
registered_models = client.search_registered_models()
print(f"\n📋 Modelos registrados disponibles: {len(registered_models)}")
for model in registered_models:
    print(f"  • {model.name}")

# Definir el nombre del modelo que queremos documentar
MODEL_NAME = "sm-job-experiment-model"  # El modelo del notebook anterior

## 2. Cargar Modelo Registrado desde MLflow

Ahora vamos a cargar la información del modelo registrado y acceder a todos sus metadatos almacenados en MLflow.

In [ ]:
# Obtener información del modelo registrado
try:
    registered_model = client.get_registered_model(MODEL_NAME)
    
    print(f"📋 Información del Modelo: {MODEL_NAME}")
    print(f"📅 Creado: {registered_model.creation_timestamp}")
    print(f"🔄 Última actualización: {registered_model.last_updated_timestamp}")
    print(f"📝 Descripción: {registered_model.description or 'Sin descripción'}")
    print(f"🏷️  Tags: {registered_model.tags}")
    
    # Obtener la versión más reciente
    latest_versions = client.get_latest_versions(MODEL_NAME, stages=["None"])
    if not latest_versions:
        # Fallback para MLflow < 3.0
        latest_versions = registered_model.latest_versions
    
    if latest_versions:
        latest_version = latest_versions[0]
        print(f"\n🔖 Versión más reciente: {latest_version.version}")
        print(f"📍 Source: {latest_version.source}")
        print(f"🏷️  Tags de versión: {latest_version.tags}")
        
        # Obtener el run ID asociado
        run_id = latest_version.run_id
        print(f"🏃 Run ID: {run_id}")
        
    else:
        print("⚠️  No se encontraron versiones del modelo")
        
except Exception as e:
    print(f"❌ Error al cargar el modelo: {str(e)}")
    print("💡 Asegúrate de que el modelo 'sm-job-experiment-model' esté registrado en MLflow")

## 3. Extraer Metadatos del Modelo

MLflow almacena automáticamente una gran cantidad de información durante el entrenamiento. Vamos a extraer todos estos metadatos para usar en nuestro Model Card.

In [ ]:
def extract_model_metadata(run_id):
    """
    Extrae todos los metadatos de un run de MLflow
    """
    try:
        # Obtener información del run
        run = client.get_run(run_id)
        run_data = run.data
        run_info = run.info
        
        metadata = {
            'run_info': {
                'run_id': run_info.run_id,
                'experiment_id': run_info.experiment_id,
                'start_time': datetime.fromtimestamp(run_info.start_time / 1000.0),
                'end_time': datetime.fromtimestamp(run_info.end_time / 1000.0) if run_info.end_time else None,
                'status': run_info.status,
                'artifact_uri': run_info.artifact_uri
            },
            'parameters': dict(run_data.params),
            'metrics': dict(run_data.metrics),
            'tags': dict(run_data.tags)
        }
        
        # Obtener información del experimento
        experiment = client.get_experiment(run_info.experiment_id)
        metadata['experiment_info'] = {
            'name': experiment.name,
            'experiment_id': experiment.experiment_id,
            'tags': experiment.tags
        }
        
        # Intentar obtener artefactos
        try:
            artifacts = client.list_artifacts(run_id)
            metadata['artifacts'] = [artifact.path for artifact in artifacts]
        except Exception as e:
            metadata['artifacts'] = []
            print(f"⚠️  No se pudieron listar artefactos: {e}")
        
        return metadata
        
    except Exception as e:
        print(f"❌ Error extrayendo metadatos: {str(e)}")
        return None

# Extraer metadatos del modelo
if 'run_id' in locals():
    print("🔍 Extrayendo metadatos del modelo...")
    model_metadata = extract_model_metadata(run_id)
    
    if model_metadata:
        print("✅ Metadatos extraídos exitosamente")
        
        print(f"\n📊 Parámetros del modelo:")
        for param, value in model_metadata['parameters'].items():
            print(f"  • {param}: {value}")
        
        print(f"\n📈 Métricas registradas:")
        for metric, value in model_metadata['metrics'].items():
            print(f"  • {metric}: {value}")
            
        print(f"\n🏷️  Tags del run:")
        for tag, value in model_metadata['tags'].items():
            print(f"  • {tag}: {value}")
            
        print(f"\n📁 Artefactos disponibles:")
        for artifact in model_metadata['artifacts']:
            print(f"  • {artifact}")
    else:
        print("❌ No se pudieron extraer los metadatos")
else:
    print("⚠️  Primero ejecuta la celda anterior para obtener el run_id")

## 4. Crear Model Card con Información de MLflow

Ahora vamos a crear una estructura de Model Card utilizando toda la información extraída de MLflow. Crearemos tanto una versión estructurada (JSON) como una versión legible (HTML/Markdown).

In [ ]:
class MLflowModelCard:
    """
    Clase para crear Model Cards a partir de metadatos de MLflow
    """
    
    def __init__(self, model_name, model_metadata, registered_model_info):
        self.model_name = model_name
        self.model_metadata = model_metadata
        self.registered_model_info = registered_model_info
        self.model_card = self._initialize_model_card()
    
    def _initialize_model_card(self):
        """Inicializa la estructura básica del Model Card"""
        return {
            "model_details": {
                "name": self.model_name,
                "version": getattr(latest_version, 'version', 'Unknown') if 'latest_version' in globals() else 'Unknown',
                "date": datetime.now().isoformat(),
                "type": "Classification Model",
                "framework": "scikit-learn",
                "mlflow_info": {
                    "run_id": self.model_metadata.get('run_info', {}).get('run_id'),
                    "experiment_id": self.model_metadata.get('run_info', {}).get('experiment_id'),
                    "experiment_name": self.model_metadata.get('experiment_info', {}).get('name'),
                    "artifact_uri": self.model_metadata.get('run_info', {}).get('artifact_uri')
                }
            },
            "intended_use": {
                "primary_intended_uses": "Clasificación de especies de flores Iris",
                "primary_intended_users": "Científicos de datos y desarrolladores de ML",
                "out_of_scope_use_cases": "No recomendado para aplicaciones críticas sin validación adicional"
            },
            "training_data": {
                "dataset": "Iris Dataset",
                "size": "150 muestras",
                "features": "4 características numéricas (longitud y ancho de sépalos y pétalos)",
                "target": "3 clases (setosa, versicolor, virginica)"
            },
            "evaluation_data": {},
            "model_parameters": self.model_metadata.get('parameters', {}),
            "metrics": self.model_metadata.get('metrics', {}),
            "ethical_considerations": {
                "sensitive_data": "No aplica - dataset público sin información sensible",
                "risks": "Riesgo mínimo - aplicación académica/educativa",
                "mitigations": "Validación cruzada y métricas de evaluación completas"
            },
            "caveats_and_recommendations": {
                "limitations": "Modelo entrenado únicamente con dataset Iris estándar",
                "recommendations": "Validar rendimiento con datos nuevos antes de uso en producción",
                "monitoring": "Monitorear drift de datos y degradación del modelo"
            }
        }
    
    def add_performance_metrics(self):
        """Añade métricas de rendimiento detalladas"""
        metrics = self.model_metadata.get('metrics', {})
        if metrics:
            self.model_card["quantitative_analyses"] = {
                "performance_metrics": metrics,
                "methodology": "Métricas calculadas durante entrenamiento con MLflow autolog",
                "evaluation_approach": "Validación automática durante entrenamiento"
            }
    
    def add_model_governance(self):
        """Añade información de gobernanza del modelo"""
        run_info = self.model_metadata.get('run_info', {})
        self.model_card["model_governance"] = {
            "approval_status": "Pending Review",
            "owner": "Data Science Team",
            "version_control": {
                "mlflow_run_id": run_info.get('run_id'),
                "training_start": str(run_info.get('start_time')),
                "training_end": str(run_info.get('end_time'))
            },
            "compliance": {
                "data_governance": "Cumple con políticas de datos internos",
                "model_risk_rating": "Low",
                "regulatory_requirements": "No aplica"
            }
        }
    
    def generate_model_card(self):
        """Genera el Model Card completo"""
        self.add_performance_metrics()
        self.add_model_governance()
        return self.model_card

# Crear Model Card si tenemos los metadatos
if 'model_metadata' in locals() and model_metadata:
    print("📋 Creando Model Card...")
    
    model_card_generator = MLflowModelCard(
        model_name=MODEL_NAME,
        model_metadata=model_metadata,
        registered_model_info=registered_model if 'registered_model' in locals() else None
    )
    
    model_card = model_card_generator.generate_model_card()
    
    print("✅ Model Card creado exitosamente")
    print(f"📊 Secciones incluidas: {list(model_card.keys())}")
else:
    print("⚠️  Primero ejecuta las celdas anteriores para obtener los metadatos")

## 5. Añadir Métricas y Parámetros

Vamos a visualizar las métricas y parámetros extraídos de MLflow de una manera más detallada y crear visualizaciones útiles para el Model Card.

In [ ]:
def visualize_model_metrics(model_card):
    """
    Crea visualizaciones de las métricas del modelo
    """
    if 'quantitative_analyses' not in model_card:
        print("⚠️  No hay métricas disponibles para visualizar")
        return
    
    metrics = model_card['quantitative_analyses']['performance_metrics']
    
    if not metrics:
        print("⚠️  No hay métricas registradas en MLflow")
        return
    
    # Crear visualización de métricas
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Gráfico de barras de métricas
    metric_names = list(metrics.keys())
    metric_values = list(metrics.values())
    
    axes[0].bar(metric_names, metric_values, color='skyblue', alpha=0.7)
    axes[0].set_title('Métricas del Modelo', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Valor')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Añadir valores en las barras
    for i, v in enumerate(metric_values):
        axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
    
    # Tabla de parámetros
    params = model_card['model_parameters']
    if params:
        param_data = [[k, v] for k, v in params.items()]
        axes[1].axis('tight')
        axes[1].axis('off')
        
        table = axes[1].table(cellText=param_data,
                            colLabels=['Parámetro', 'Valor'],
                            cellLoc='center',
                            loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.5)
        axes[1].set_title('Parámetros del Modelo', fontsize=14, fontweight='bold')
    else:
        axes[1].text(0.5, 0.5, 'No hay parámetros\ndisponibles', 
                    ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('Parámetros del Modelo', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return fig

def create_model_summary_table(model_card):
    """
    Crea una tabla resumen del modelo
    """
    summary_data = []
    
    # Información básica
    model_details = model_card.get('model_details', {})
    summary_data.extend([
        ['Nombre del Modelo', model_details.get('name', 'N/A')],
        ['Versión', model_details.get('version', 'N/A')],
        ['Tipo', model_details.get('type', 'N/A')],
        ['Framework', model_details.get('framework', 'N/A')],
        ['Fecha de Creación', model_details.get('date', 'N/A')[:10]]
    ])
    
    # Información de MLflow
    mlflow_info = model_details.get('mlflow_info', {})
    summary_data.extend([
        ['MLflow Run ID', mlflow_info.get('run_id', 'N/A')[:8] + '...' if mlflow_info.get('run_id') else 'N/A'],
        ['Experimento', mlflow_info.get('experiment_name', 'N/A')]
    ])
    
    # Crear DataFrame para mejor visualización
    df_summary = pd.DataFrame(summary_data, columns=['Campo', 'Valor'])
    
    print("📋 Resumen del Modelo:")
    print("=" * 50)
    for _, row in df_summary.iterrows():
        print(f"{row['Campo']:<20}: {row['Valor']}")
    
    return df_summary

# Ejecutar visualizaciones si tenemos el model card
if 'model_card' in locals():
    print("📊 Generando visualizaciones del modelo...")
    
    # Crear tabla resumen
    summary_df = create_model_summary_table(model_card)
    
    print("\n")
    
    # Crear visualizaciones de métricas
    fig = visualize_model_metrics(model_card)
    
else:
    print("⚠️  Primero ejecuta las celdas anteriores para crear el model card")

## 6. Generar Documentación del Modelo

Ahora vamos a crear una documentación completa del modelo en formato HTML que incluya toda la información extraída de MLflow.

In [ ]:
# Template HTML para el Model Card
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Model Card - {{ model_card.model_details.name }}</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f8f9fa;
        }
        .header {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 30px;
            border-radius: 10px;
            margin-bottom: 30px;
        }
        .section {
            background: white;
            padding: 25px;
            margin-bottom: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .section h2 {
            color: #333;
            border-bottom: 2px solid #667eea;
            padding-bottom: 10px;
        }
        .grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
        }
        .metric-card {
            background: #f8f9fa;
            padding: 15px;
            border-radius: 5px;
            border-left: 4px solid #667eea;
        }
        .tag {
            background: #e9ecef;
            padding: 5px 10px;
            border-radius: 15px;
            display: inline-block;
            margin: 2px;
            font-size: 0.9em;
        }
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            text-align: left;
            padding: 12px;
            border-bottom: 1px solid #ddd;
        }
        th {
            background-color: #f8f9fa;
            font-weight: bold;
        }
        .mlflow-info {
            background: #e8f4f8;
            padding: 15px;
            border-radius: 5px;
            border-left: 4px solid #17a2b8;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>🤖 Model Card</h1>
        <h2>{{ model_card.model_details.name }}</h2>
        <p><strong>Versión:</strong> {{ model_card.model_details.version }} | 
           <strong>Framework:</strong> {{ model_card.model_details.framework }} | 
           <strong>Fecha:</strong> {{ model_card.model_details.date[:10] }}</p>
    </div>

    <div class="section">
        <h2>🎯 Detalles del Modelo</h2>
        <div class="grid">
            <div>
                <h3>Información General</h3>
                <table>
                    <tr><td><strong>Nombre</strong></td><td>{{ model_card.model_details.name }}</td></tr>
                    <tr><td><strong>Tipo</strong></td><td>{{ model_card.model_details.type }}</td></tr>
                    <tr><td><strong>Framework</strong></td><td>{{ model_card.model_details.framework }}</td></tr>
                    <tr><td><strong>Versión</strong></td><td>{{ model_card.model_details.version }}</td></tr>
                </table>
            </div>
            <div class="mlflow-info">
                <h3>🔗 Información de MLflow</h3>
                <p><strong>Run ID:</strong> <code>{{ model_card.model_details.mlflow_info.run_id[:12] }}...</code></p>
                <p><strong>Experimento:</strong> {{ model_card.model_details.mlflow_info.experiment_name }}</p>
                <p><strong>Artifact URI:</strong> <small>{{ model_card.model_details.mlflow_info.artifact_uri[:50] }}...</small></p>
            </div>
        </div>
    </div>

    <div class="section">
        <h2>📊 Parámetros del Modelo</h2>
        {% if model_card.model_parameters %}
        <div class="grid">
            {% for param, value in model_card.model_parameters.items() %}
            <div class="metric-card">
                <strong>{{ param }}</strong><br>
                <span style="font-size: 1.2em; color: #667eea;">{{ value }}</span>
            </div>
            {% endfor %}
        </div>
        {% else %}
        <p>No hay parámetros registrados.</p>
        {% endif %}
    </div>

    {% if model_card.quantitative_analyses %}
    <div class="section">
        <h2>📈 Métricas de Rendimiento</h2>
        <div class="grid">
            {% for metric, value in model_card.quantitative_analyses.performance_metrics.items() %}
            <div class="metric-card">
                <strong>{{ metric }}</strong><br>
                <span style="font-size: 1.2em; color: #28a745;">{{ "%.4f"|format(value) }}</span>
            </div>
            {% endfor %}
        </div>
        <p><em>Metodología:</em> {{ model_card.quantitative_analyses.methodology }}</p>
    </div>
    {% endif %}

    <div class="section">
        <h2>🎯 Uso Previsto</h2>
        <div class="grid">
            <div>
                <h3>Usos Principales</h3>
                <p>{{ model_card.intended_use.primary_intended_uses }}</p>
                
                <h3>Usuarios Objetivo</h3>
                <p>{{ model_card.intended_use.primary_intended_users }}</p>
            </div>
            <div>
                <h3>⚠️ Casos Fuera del Alcance</h3>
                <p>{{ model_card.intended_use.out_of_scope_use_cases }}</p>
            </div>
        </div>
    </div>

    <div class="section">
        <h2>📋 Datos de Entrenamiento</h2>
        <table>
            <tr><td><strong>Dataset</strong></td><td>{{ model_card.training_data.dataset }}</td></tr>
            <tr><td><strong>Tamaño</strong></td><td>{{ model_card.training_data.size }}</td></tr>
            <tr><td><strong>Características</strong></td><td>{{ model_card.training_data.features }}</td></tr>
            <tr><td><strong>Variable Objetivo</strong></td><td>{{ model_card.training_data.target }}</td></tr>
        </table>
    </div>

    {% if model_card.model_governance %}
    <div class="section">
        <h2>🏛️ Gobernanza del Modelo</h2>
        <div class="grid">
            <div>
                <h3>Control de Versiones</h3>
                <div class="mlflow-info">
                    <p><strong>MLflow Run ID:</strong> <code>{{ model_card.model_governance.version_control.mlflow_run_id[:12] }}...</code></p>
                    <p><strong>Inicio Entrenamiento:</strong> {{ model_card.model_governance.version_control.training_start[:19] }}</p>
                    <p><strong>Fin Entrenamiento:</strong> {{ model_card.model_governance.version_control.training_end[:19] }}</p>
                </div>
            </div>
            <div>
                <h3>Cumplimiento</h3>
                <p><strong>Estado:</strong> <span class="tag">{{ model_card.model_governance.approval_status }}</span></p>
                <p><strong>Propietario:</strong> {{ model_card.model_governance.owner }}</p>
                <p><strong>Rating de Riesgo:</strong> <span class="tag">{{ model_card.model_governance.compliance.model_risk_rating }}</span></p>
            </div>
        </div>
    </div>
    {% endif %}

    <div class="section">
        <h2>⚖️ Consideraciones Éticas</h2>
        <div class="grid">
            <div>
                <h3>Datos Sensibles</h3>
                <p>{{ model_card.ethical_considerations.sensitive_data }}</p>
                
                <h3>Riesgos</h3>
                <p>{{ model_card.ethical_considerations.risks }}</p>
            </div>
            <div>
                <h3>Mitigaciones</h3>
                <p>{{ model_card.ethical_considerations.mitigations }}</p>
            </div>
        </div>
    </div>

    <div class="section">
        <h2>⚠️ Limitaciones y Recomendaciones</h2>
        <div class="grid">
            <div>
                <h3>Limitaciones</h3>
                <p>{{ model_card.caveats_and_recommendations.limitations }}</p>
            </div>
            <div>
                <h3>Recomendaciones</h3>
                <p>{{ model_card.caveats_and_recommendations.recommendations }}</p>
                
                <h3>Monitoreo</h3>
                <p>{{ model_card.caveats_and_recommendations.monitoring }}</p>
            </div>
        </div>
    </div>

    <footer style="text-align: center; margin-top: 40px; padding: 20px; background: #e9ecef; border-radius: 5px;">
        <p><em>Model Card generado automáticamente desde MLflow el {{ model_card.model_details.date[:19] }}</em></p>
    </footer>
</body>
</html>
"""

def generate_html_model_card(model_card, output_path="model_card.html"):
    """
    Genera un Model Card en formato HTML
    """
    try:
        template = Template(HTML_TEMPLATE)
        html_content = template.render(model_card=model_card)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        print(f"✅ Model Card HTML generado: {output_path}")
        return output_path
        
    except Exception as e:
        print(f"❌ Error generando HTML: {str(e)}")
        return None

def generate_json_model_card(model_card, output_path="model_card.json"):
    """
    Guarda el Model Card en formato JSON
    """
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(model_card, f, indent=2, ensure_ascii=False, default=str)
        
        print(f"✅ Model Card JSON generado: {output_path}")
        return output_path
        
    except Exception as e:
        print(f"❌ Error generando JSON: {str(e)}")
        return None

# Generar documentación si tenemos el model card
if 'model_card' in locals():
    print("📝 Generando documentación del modelo...")
    
    # Crear directorio para guardar los archivos
    os.makedirs("../model_cards", exist_ok=True)
    
    # Generar archivos
    html_path = generate_html_model_card(model_card, "../model_cards/iris_model_card.html")
    json_path = generate_json_model_card(model_card, "../model_cards/iris_model_card.json")
    
    if html_path:
        print(f"🌐 Abre el archivo HTML en tu navegador: {html_path}")
        
else:
    print("⚠️  Primero ejecuta las celdas anteriores para crear el model card")

## 7. Publicar Model Card

Finalmente, vamos a mostrar cómo vincular el Model Card generado con el modelo registrado en MLflow para mantener la trazabilidad completa.

In [ ]:
def link_model_card_to_mlflow(model_name, model_card_paths, model_version=None):
    """
    Vincula el Model Card con el modelo registrado en MLflow usando tags
    """
    try:
        # Preparar tags con información del Model Card
        model_card_tags = {
            "model_card.html_path": model_card_paths.get('html', ''),
            "model_card.json_path": model_card_paths.get('json', ''),
            "model_card.generated_at": datetime.now().isoformat(),
            "model_card.version": "1.0",
            "documentation.status": "completed",
            "governance.reviewed": "pending"
        }
        
        if model_version:
            # Actualizar tags en una versión específica
            client.set_model_version_tag(model_name, model_version, "model_card.linked", "true")
            for tag_key, tag_value in model_card_tags.items():
                client.set_model_version_tag(model_name, model_version, tag_key, tag_value)
            print(f"✅ Model Card vinculado a {model_name} versión {model_version}")
        else:
            # Actualizar tags en el modelo registrado
            for tag_key, tag_value in model_card_tags.items():
                client.set_registered_model_tag(model_name, tag_key, tag_value)
            print(f"✅ Model Card vinculado a {model_name}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error vinculando Model Card: {str(e)}")
        return False

def create_model_registry_summary():
    """
    Crea un resumen del estado del modelo en el registry
    """
    try:
        # Obtener información actualizada del modelo
        updated_model = client.get_registered_model(MODEL_NAME)
        latest_versions = client.get_latest_versions(MODEL_NAME, stages=["None"])
        
        if latest_versions:
            latest_version = latest_versions[0]
            
            print("📋 Resumen del Modelo en MLflow Registry")
            print("=" * 60)
            print(f"🏷️  Nombre: {updated_model.name}")
            print(f"📅 Creado: {datetime.fromtimestamp(updated_model.creation_timestamp/1000)}")
            print(f"🔄 Actualizado: {datetime.fromtimestamp(updated_model.last_updated_timestamp/1000)}")
            print(f"🔖 Versión actual: {latest_version.version}")
            
            print(f"\n🏷️  Tags del modelo:")
            for tag, value in updated_model.tags.items():
                if tag.startswith('model_card'):
                    print(f"  • {tag}: {value}")
            
            print(f"\n🏷️  Tags de la versión:")
            for tag, value in latest_version.tags.items():
                if tag.startswith('model_card') or tag.startswith('documentation'):
                    print(f"  • {tag}: {value}")
                    
            # Verificar si el Model Card está vinculado
            has_model_card = any(tag.startswith('model_card') for tag in updated_model.tags.keys())
            if has_model_card:
                print(f"\n✅ Model Card vinculado correctamente")
            else:
                print(f"\n⚠️  Model Card no vinculado")
        
    except Exception as e:
        print(f"❌ Error obteniendo resumen: {str(e)}")

def generate_model_lineage_report():
    """
    Genera un reporte de lineage del modelo
    """
    if 'model_metadata' not in locals() or not model_metadata:
        print("⚠️  No hay metadatos disponibles para generar el reporte")
        return
    
    lineage_report = {
        "model_info": {
            "name": MODEL_NAME,
            "version": getattr(latest_version, 'version', 'Unknown') if 'latest_version' in globals() else 'Unknown',
            "mlflow_run_id": model_metadata.get('run_info', {}).get('run_id'),
            "experiment_id": model_metadata.get('run_info', {}).get('experiment_id')
        },
        "data_lineage": {
            "training_data": "Iris Dataset",
            "data_source": "UCI Machine Learning Repository",
            "preprocessing": "Standard MLflow autolog preprocessing"
        },
        "model_lineage": {
            "algorithm": "Decision Tree Classifier",
            "framework": "scikit-learn",
            "training_date": str(model_metadata.get('run_info', {}).get('start_time')),
            "parameters": model_metadata.get('parameters', {}),
            "metrics": model_metadata.get('metrics', {})
        },
        "governance": {
            "model_card_generated": True,
            "documentation_complete": True,
            "approval_status": "pending_review",
            "responsible_team": "Data Science Team"
        }
    }
    
    print("🔍 Reporte de Lineage del Modelo")
    print("=" * 50)
    print(json.dumps(lineage_report, indent=2, default=str))
    
    # Guardar reporte
    os.makedirs("../model_cards", exist_ok=True)
    with open("../model_cards/model_lineage_report.json", 'w') as f:
        json.dump(lineage_report, f, indent=2, default=str)
    
    print(f"\n💾 Reporte guardado en: ../model_cards/model_lineage_report.json")

# Ejecutar funciones de publicación
if 'model_card' in locals() and 'html_path' in locals() and 'json_path' in locals():
    print("🚀 Publicando Model Card...")
    
    # Vincular Model Card con MLflow
    model_card_paths = {
        'html': html_path,
        'json': json_path
    }
    
    success = link_model_card_to_mlflow(
        MODEL_NAME, 
        model_card_paths, 
        model_version=getattr(latest_version, 'version', None) if 'latest_version' in globals() else None
    )
    
    if success:
        print("\n")
        # Mostrar resumen actualizado
        create_model_registry_summary()
        
        print("\n")
        # Generar reporte de lineage
        generate_model_lineage_report()
        
        print("\n🎉 ¡Model Card publicado exitosamente!")
        print("📁 Archivos generados:")
        print(f"  • Model Card HTML: {html_path}")
        print(f"  • Model Card JSON: {json_path}")
        print(f"  • Lineage Report: ../model_cards/model_lineage_report.json")
        
else:
    print("⚠️  Primero ejecuta las celdas anteriores para generar el Model Card")

## Resumen y Próximos Pasos

### ✅ Lo que hemos logrado:

1. **Extracción automatizada**: Hemos extraído todos los metadatos, parámetros y métricas almacenados por MLflow durante el entrenamiento
2. **Model Card completo**: Generamos documentación comprehensiva incluyendo aspectos técnicos, éticos y de gobernanza
3. **Visualizaciones**: Creamos gráficos y tablas para mejor comprensión del modelo
4. **Múltiples formatos**: Generamos tanto JSON estructurado como HTML legible
5. **Trazabilidad**: Vinculamos el Model Card directamente con el modelo en MLflow Registry
6. **Lineage completo**: Documentamos el linaje completo del modelo y datos

### 🔄 Flujo de trabajo recomendado:

1. **Entrenar modelo** con MLflow (como en el notebook anterior)
2. **Ejecutar este notebook** para generar Model Card automáticamente
3. **Revisar y personalizar** la documentación según necesidades específicas
4. **Aprobar y publicar** el Model Card para uso en producción

### 📈 Beneficios clave:

- **Automatización**: No hay duplicación manual de información
- **Consistencia**: La documentación siempre refleja el estado actual del modelo
- **Gobernanza**: Mejor control y auditabilía de modelos
- **Colaboración**: Documentación clara para todos los stakeholders
- **Cumplimiento**: Facilita auditorías y cumplimiento regulatorio

### 🚀 Extensiones posibles:

- Integrar con sistemas de CI/CD para generar Model Cards automáticamente
- Añadir validaciones adicionales de fairness y bias
- Conectar con sistemas de monitoreo en producción
- Integrar con herramientas de governance corporativas